# PSDAT tour — renewables in the dynamics classroom
Run cell by cell. Requires `numpy` and `matplotlib`; run from the `python/` folder.

In [ ]:
import sys, numpy as np
import cases
from system import System
from linearize import linearize, print_modes, participation, dominant_mode
from simulate import simulate, cloud_profile, gust_profile
import matplotlib.pyplot as plt
%matplotlib inline

## 1. The all-synchronous base case
PSDAT reproduces the published PSDAT modes (1.4443 Hz / 7.16 %, 2.3721 Hz / 7.73 %).

In [ ]:
c9 = cases.ieee9()
s = System(c9, ['SG','SG','SG'])
R = linearize(s, verbose=True)
print_modes(R, label='IEEE9 all-SG')

## 2. Re-equip the grid in one line
A grid-forming battery at G2 and a PV plant at G3 — same network, new physics.

In [ ]:
s = System(c9, ['SG','BESS-GFM','PV-GFL'])
R = linearize(s)
print_modes(R, label='SG + BESS-GFM + PV-GFL')
nm = s.state_names()
dm = dominant_mode(R, [j for j,x in enumerate(nm) if x in ('Vdc3','xdc3')], fband=(0.5,20))
print(f"DC-link mode: {dm['f']:.2f} Hz, {dm['zeta']:.0f}% damping")

## 3. Source-side disturbance: the cloud
The third disturbance class — the renewable resource itself.

In [ ]:
T, X, Z = simulate(s, tsim=18, dt=2e-3, G_prof={2: cloud_profile(2.0, depth=0.6)})
fig, ax = plt.subplots(2, 1, figsize=(7,5), sharex=True)
ax[0].plot(T, s.unit_state(X,2,'Vdc')); ax[0].set_ylabel('$V_{dc}$ (pu)'); ax[0].grid(True)
ax[1].plot(T, s.coi_freq(X)); ax[1].set_ylabel('COI f (Hz)'); ax[1].set_xlabel('t (s)'); ax[1].grid(True)
plt.show()

## 4. Wind: gust through a DFIG
Watch the rotor buffer the gust (slip moves, output barely does).

In [ ]:
s = System(c9, ['SG','SG','WT3'])
T, X, Z = simulate(s, tsim=16, dt=2e-3, vw_prof={2: gust_profile(2.0, A=0.15, base=0.9)})
fig, ax = plt.subplots(2, 1, figsize=(7,5), sharex=True)
ax[0].plot(T, s.unit_state(X,2,'wt'), label='turbine speed')
ax[0].plot(T, 1-s.unit_state(X,2,'slip'), label='generator speed')
ax[0].legend(); ax[0].set_ylabel('speed (pu)'); ax[0].grid(True)
ax[1].plot(T, s.coi_freq(X)); ax[1].set_ylabel('COI f (Hz)'); ax[1].set_xlabel('t (s)'); ax[1].grid(True)
plt.show()

## 5. Battery fast frequency response
Sweep the FFR droop gain and watch the nadir recover.

In [ ]:
for Kf in (0, 25, 50):
    s = System(c9, ['SG','BESS-GFL','GFL'], [None, dict(Kf=Kf), None])
    T, X, Z = simulate(s, tsim=10, dt=2e-3, t_dist=1.0, dPload={7:0.15})
    fc = s.coi_freq(X)
    plt.plot(T, fc, label=f'$K_f$={Kf}  (nadir {fc[T>=1].min():.3f} Hz)')
plt.grid(True); plt.legend(); plt.xlabel('t (s)'); plt.ylabel('COI f (Hz)'); plt.show()

## 6. Control design: damp the Kundur inter-area mode from a battery
Residue-based POD, verified on the exact closed loop.

In [ ]:
import design as D
ck = cases.kundur2a()
s = System(ck, ['SG','BESS-GFM','SG','SG'], [None, dict(cases.GFM_K, Eh=1.0), None, None])
R = linearize(s)
lam0 = sorted([l for l in R['ev'] if 0.3 < l.imag/2/np.pi < 0.9], key=lambda l: -l.real/abs(l))[0]
B = D.input_matrix(R, units=[1]); C = D.output_matrix(R, D.speed_output(s, 1))
res, lam, i = D.residues(R, B, C, lam0)
pod = D.pod_design(res[0], lam, zeta_target=0.20)
ev = np.linalg.eig(D.closed_loop(R, B[:,0], C, pod))[0]
z1, lam1 = D.damping_of(ev, lam)
print(f"inter-area damping: {-lam.real/abs(lam)*100:.1f}% -> {z1:.1f}% (target 20%)")
plt.plot(R['ev'].real, R['ev'].imag, 'x', label='open loop')
plt.plot(ev.real, ev.imag, 'o', mfc='none', label='with POD')
plt.xlim(-3.5, 0.5); plt.ylim(0, 12); plt.grid(True); plt.legend()
plt.xlabel('Re (1/s)'); plt.ylabel('Im (rad/s)'); plt.show()

## Where next
`python3 run_scenario.py` lists all guided studies; `docs/MANUAL.md` documents every model with its textbook source; `docs/PROBLEMS.md` has course assignments.